In [1]:
import os

In [2]:
%pwd

'/Users/gianlucadebonis/Desktop/Courses/MLOpsBootcamp/data-science-project/research'

In [3]:
os.chdir('../')
%pwd

'/Users/gianlucadebonis/Desktop/Courses/MLOpsBootcamp/data-science-project'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [5]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(self, 
                 config_filepath = CONFIG_FILEPATH,
                 params_filepath = PARAMS_FILEPATH,
                 schema_filepath = SCHEMA_FILEPATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])

        return DataIngestionConfig(
            root_dir = config.root_dir,
            source_URL = config.source_URL,
            local_data_file = config.local_data_file,
            unzip_dir = config.unzip_dir
        )

In [7]:
import urllib.request as request
from src.datascience import logger
import zipfile

In [8]:
# Component - Data Ingestion
import requests

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    # Downloading the zip file
    def download_file(self):
        if not os.path.exists(self.config.local_data_file):
            filename, headers = request.urlretrieve(
                url = self.config.source_URL,
                filename = self.config.local_data_file
            )
            logger.info(f'{filename} download! With the following info: \n{headers}')
            logger.info(f'File already exists.')
    
    def extract_zip_file(self):
        """
        Extracts the zip file into the data directory
        """
        unzip_path = self.config.unzip_dir
        os.makedirs(unzip_path, exist_ok = True)
        with zipfile.ZipFile(self.config.local_data_file, 'r') as zip_ref:
            zip_ref.extractall(unzip_path)

In [9]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config = data_ingestion_config)
    data_ingestion.download_file()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e

[2026-05-28 17:13:28,412: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-05-28 17:13:28,415: INFO: common: yaml file: params.yaml loaded successfully]
[2026-05-28 17:13:28,416: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-05-28 17:13:28,417: INFO: common: Created directory at: artifacts]
[2026-05-28 17:13:28,417: INFO: common: Created directory at: artifacts/data_ingestion]
